# Task 8 — Track C (Medical dialogues): Entity & event extraction

Цель: извлечь из медицинских диалогов сущности (SYMPTOM, DIAGNOSIS, MEDICATION, DOSAGE, DURATION, SIDE_EFFECT) с помощью локально запускаемых моделей и сравнить скорость/ресурсы.

В ноутбуке: 
- загрузка датасета `omi-health/medical-dialogue-to-soap-summary` и подвыборка
- IE через LLM с JSON-выходом
- batch processing и замеры tokens/sec
- упрощённая оценка precision/recall на небольшом вручную размеченном наборе (synthetic gold)

In [1]:
import os
import re
import json
import time
import math
from dataclasses import dataclass
from typing import Any, Dict, List, Optional, Tuple

import numpy as np
import pandas as pd
import psutil
from tqdm.auto import tqdm

/home/zodiac/stadygit/LLM-Driven-Development/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
import torch
from datasets import load_dataset
from transformers import AutoModelForCausalLM, AutoTokenizer

## Конфигурация

По умолчанию ноутбук использует небольшие instruct-модели, которые реально запустить локально на CPU/GPU.
Если вы хотите строго следовать списку из задания (BioMistral/Mistral/Llama-2), можно заменить `MODEL_CANDIDATES` на доступные вам веса.

Примечание: некоторые модели на Hugging Face могут быть gated (потребуют принятия лицензии). В этом случае просто выберите другие открытые модели.

In [3]:
SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)

DATASET_ID = "omi-health/medical-dialogue-to-soap-summary"
SPLIT = "train"
N_DEMO_ROWS = 120

# 2-3 модели для сравнения (можно заменить).
# Держим список коротким: некоторые веса большие.
MODEL_CANDIDATES = [
    # Обычно доступны и относительно компактны
    "Qwen/Qwen2.5-1.5B-Instruct",
    "microsoft/Phi-3-mini-4k-instruct",
    # Опционально более тяжёлая мед-ориентированная модель (если доступна)
    # "BioMistral/BioMistral-7B",
]

MAX_INPUT_CHARS = 2500
MAX_NEW_TOKENS = 220
TEMPERATURE = 0.0

BATCH_SIZE = 4

device = "cuda" if torch.cuda.is_available() else "cpu"
device

'cuda'

## Загрузка данных

Берём подвыборку для demo (100–200 строк). В тексте используем диалог как основной источник фактов.

In [4]:
ds = load_dataset(DATASET_ID, split=SPLIT)
len(ds)

9250

In [5]:
# Посмотрим на поля, чтобы корректно собрать текст
ds[0].keys()

dict_keys(['dialogue', 'soap', 'prompt', 'messages', 'messages_nosystem'])

In [6]:
sample0 = ds[0]
{k: (str(sample0[k])[:200] + ('...' if len(str(sample0[k])) > 200 else '')) for k in sample0.keys()}

{'dialogue': "Doctor: Hello, how can I help you today?\nPatient: My son has been having some issues with speech and development. He's 13 years old now.\nDoctor: I see. Can you tell me more about his symptoms? Does he...",
 'soap': "S: The patient's mother reports that her 13-year-old son has mild to moderate speech and developmental delays and has been diagnosed with attention deficit disorder. She denies any issues with muscle ...",
 'prompt': "Create a Medical SOAP note summary from the dialogue, following these guidelines:\n    S (Subjective): Summarize the patient's reported symptoms, including chief complaint and relevant history. Rely on...",
 'messages': "[{'role': 'system', 'content': 'You are an expert medical professor assisting in the creation of medically accurate SOAP summaries. Please ensure the response follows the structured format: S:, O:, A:...",
 'messages_nosystem': '[{\'role\': \'user\', \'content\': "You are an expert medical professor assisting in the creation of

In [7]:
def _pick_dialogue_field(example: Dict[str, Any]) -> str:
    # На практике поле может называться по-разному; подстраховываемся.
    for key in ["dialogue", "conversation", "text", "dialog", "transcript"]:
        if key in example and example[key]:
            return str(example[key])
    # fallback: соберём все строковые поля
    parts = []
    for k, v in example.items():
        if isinstance(v, str) and v.strip():
            parts.append(f"{k}: {v}")
    return "\n".join(parts)

def normalize_text(s: str, max_chars: int = MAX_INPUT_CHARS) -> str:
    s = re.sub(r"\s+", " ", s).strip()
    if len(s) <= max_chars:
        return s
    return s[:max_chars].rstrip() + "..."

rows = []
for i in range(N_DEMO_ROWS):
    ex = ds[i]
    dialog = normalize_text(_pick_dialogue_field(ex))
    rows.append({"id": i, "dialogue": dialog, "raw": ex})

df = pd.DataFrame(rows)
df.head(3)

,id,dialogue,raw
0,0,"Doctor: Hello, how can I help you today? Patie...","{'dialogue': 'Doctor: Hello, how can I help yo..."
1,1,"Doctor: Hello, what brings you in today? Patie...","{'dialogue': 'Doctor: Hello, what brings you i..."
2,2,"Doctor: Hello, how can I help you today? Patie...","{'dialogue': 'Doctor: Hello, how can I help yo..."


## Схема извлечения и промпт

Формат вывода фиксируем как JSON со строго заданными ключами. Это упрощает постобработку и оценку качества.

In [8]:
SCHEMA_KEYS = [
    "symptoms",
    "diagnoses",
    "medications",
    "dosages",
    "durations",
    "side_effects",
]

SYSTEM_INSTRUCTION = (
    "You are an information extraction system for medical dialogues. "
    "Extract entities and events from the dialogue. "
    "Return ONLY a valid JSON object with the exact keys: "
    + ", ".join(SCHEMA_KEYS)
)

PROMPT_TEMPLATE = (
    "Task: extract entities from the medical dialogue.\n"
    "Rules:\n"
    "- Output must be a single JSON object and nothing else.\n"
    "- Each value is a list of strings. If nothing found, return an empty list.\n"
    "- Do not add new keys.\n"
    "- Keep strings short, as they appear in text.\n"
    "Dialogue:\n{dialogue}\n"
    "JSON:\n"
)


def build_prompt(dialogue: str) -> str:
    return PROMPT_TEMPLATE.format(dialogue=dialogue)


In [9]:
def safe_json_load(s: str) -> Optional[Dict[str, Any]]:
    # Часто модель добавляет префиксы/суффиксы — попробуем вытащить JSON-блок.
    s = s.strip()
    if not s:
        return None
    m = re.search(r"\{.*\}", s, flags=re.DOTALL)
    if not m:
        return None
    blob = m.group(0)
    try:
        obj = json.loads(blob)
    except json.JSONDecodeError:
        return None
    if not isinstance(obj, dict):
        return None
    # Нормализуем ключи
    out = {}
    for k in SCHEMA_KEYS:
        v = obj.get(k, [])
        if v is None:
            v = []
        if isinstance(v, str):
            v = [v]
        if not isinstance(v, list):
            v = []
        out[k] = [str(x).strip() for x in v if str(x).strip()]
    return out

## Загрузка модели и генерация

Сделаем единый загрузчик, который может (опционально) включать 4-bit через bitsandbytes. Если `bitsandbytes` не установлен или нет CUDA — используем обычную загрузку.

In [10]:
@dataclass
class LoadedModel:
    model_id: str
    tokenizer: Any
    model: Any
    quantization: str

def load_llm(model_id: str, quantization: str = "none") -> LoadedModel:
    # quantization: 'none' | '4bit'
    quantization = quantization.lower().strip()

    tokenizer = AutoTokenizer.from_pretrained(model_id, use_fast=True)
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token

    common_kwargs = {"device_map": "auto" if device == "cuda" else None}

    if quantization == "4bit":
        if device != "cuda":
            raise RuntimeError("4bit quantization requires CUDA")
        try:
            from transformers import BitsAndBytesConfig
        except Exception as e:
            raise RuntimeError("BitsAndBytesConfig not available; upgrade transformers") from e

        try:
            import bitsandbytes  # noqa: F401
        except Exception as e:
            raise RuntimeError("bitsandbytes is not installed") from e

        bnb_config = BitsAndBytesConfig(
            load_in_4bit=True,
            bnb_4bit_compute_dtype=torch.float16,
            bnb_4bit_use_double_quant=True,
            bnb_4bit_quant_type="nf4",
        )
        model = AutoModelForCausalLM.from_pretrained(
            model_id,
            quantization_config=bnb_config,
            torch_dtype=torch.float16,
            **common_kwargs,
        )
        return LoadedModel(model_id=model_id, tokenizer=tokenizer, model=model, quantization="4bit")

    # non-quantized
    torch_dtype = torch.float16 if device == "cuda" else torch.float32
    model = AutoModelForCausalLM.from_pretrained(
        model_id,
        torch_dtype=torch_dtype,
        **common_kwargs,
    )
    if device == "cpu":
        model.to(device)
    model.eval()
    return LoadedModel(model_id=model_id, tokenizer=tokenizer, model=model, quantization="none")

In [11]:
def format_chat_or_plain(tokenizer, system: str, user: str) -> str:
    # Универсальный путь: если есть chat template — используем его, иначе склеиваем вручную.
    if hasattr(tokenizer, "apply_chat_template") and tokenizer.chat_template is not None:
        messages = []
        if system:
            messages.append({"role": "system", "content": system})
        messages.append({"role": "user", "content": user})
        return tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)

    parts = []
    if system:
        parts.append(f"SYSTEM: {system}")
    parts.append(f"USER: {user}")
    parts.append("ASSISTANT:")
    return "\n".join(parts)

@torch.no_grad()
def generate_batch(loaded: LoadedModel, prompts: List[str]) -> Tuple[List[str], int, float]:
    tok = loaded.tokenizer
    model = loaded.model

    enc = tok(
        prompts,
        return_tensors="pt",
        padding=True,
        truncation=True,
        max_length=tok.model_max_length if tok.model_max_length and tok.model_max_length < 4096 else 4096,
    )
    enc = {k: v.to(model.device) for k, v in enc.items()}

    start = time.perf_counter()
    out = model.generate(
        **enc,
        max_new_tokens=MAX_NEW_TOKENS,
        do_sample=(TEMPERATURE > 0),
        temperature=TEMPERATURE if TEMPERATURE > 0 else None,
        pad_token_id=tok.pad_token_id,
        eos_token_id=tok.eos_token_id,
    )
    end = time.perf_counter()

    # generated tokens = total - input
    gen_tokens = int(out.shape[1] - enc["input_ids"].shape[1])

    texts = tok.batch_decode(out, skip_special_tokens=True)
    return texts, gen_tokens, (end - start)

## Демонстрация извлечения на нескольких примерах

In [12]:
def extract_entities_for_texts(loaded: LoadedModel, texts: List[str], batch_size: int = BATCH_SIZE) -> pd.DataFrame:
    records = []
    total_gen_tokens = 0
    total_time = 0.0

    for i in tqdm(range(0, len(texts), batch_size), desc=f"IE {loaded.model_id} ({loaded.quantization})"):
        batch = texts[i:i+batch_size]
        prompts = [format_chat_or_plain(loaded.tokenizer, SYSTEM_INSTRUCTION, build_prompt(t)) for t in batch]
        outputs, gen_tokens, dt = generate_batch(loaded, prompts)
        total_gen_tokens += gen_tokens
        total_time += dt

        for src_text, full_out in zip(batch, outputs):
            parsed = safe_json_load(full_out)
            records.append({"text": src_text, "raw_output": full_out, "parsed": parsed})

    tokens_per_sec = total_gen_tokens / max(total_time, 1e-9)
    df_out = pd.DataFrame(records)
    df_out.attrs["generated_tokens"] = total_gen_tokens
    df_out.attrs["generation_time_sec"] = total_time
    df_out.attrs["tokens_per_sec"] = tokens_per_sec
    return df_out

demo_texts = df["dialogue"].head(6).tolist()
demo_texts[0][:350]

"Doctor: Hello, how can I help you today? Patient: My son has been having some issues with speech and development. He's 13 years old now. Doctor: I see. Can you tell me more about his symptoms? Does he have any issues with muscle tone or hypotonia? Patient: No, he doesn't have hypotonia. But he has mild to moderate speech and developmental delay, an"

In [13]:
# Выберем первую доступную модель
model_id = MODEL_CANDIDATES[0]
loaded0 = load_llm(model_id, quantization="none")
loaded0

`torch_dtype` is deprecated! Use `dtype` instead!
Loading weights: 100%|██████████| 338/338 [00:00<00:00, 355.86it/s, Materializing param=model.norm.weight]                              


LoadedModel(model_id='Qwen/Qwen2.5-1.5B-Instruct', tokenizer=Qwen2Tokenizer(name_or_path='Qwen/Qwen2.5-1.5B-Instruct', vocab_size=151643, model_max_length=131072, padding_side='right', truncation_side='right', special_tokens={'eos_token': '<|im_end|>', 'pad_token': '<|endoftext|>'}, added_tokens_decoder={
	151643: AddedToken("<|endoftext|>", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	151644: AddedToken("<|im_start|>", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	151645: AddedToken("<|im_end|>", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	151646: AddedToken("<|object_ref_start|>", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	151647: AddedToken("<|object_ref_end|>", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	151648: AddedToken("<|box_start|>", rstrip=False, lstrip=False, single_word=False, normalized=False

In [14]:
demo_res = extract_entities_for_texts(loaded0, demo_texts, batch_size=2)
demo_res[["text", "parsed"]].head(3)

IE Qwen/Qwen2.5-1.5B-Instruct (none):   0%|          | 0/3 [00:00<?, ?it/s]The following generation flags are not valid and may be ignored: ['top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
A decoder-only architecture is being used, but right-padding was detected! For correct generation results, please set `padding_side='left'` when initializing the tokenizer.
IE Qwen/Qwen2.5-1.5B-Instruct (none): 100%|██████████| 3/3 [00:07<00:00,  2.34s/it]


,text,parsed
0,"Doctor: Hello, how can I help you today? Patie...","{'symptoms': [], 'diagnoses': ['attention defi..."
1,"Doctor: Hello, what brings you in today? Patie...","{'symptoms': ['weakness in lower extremities',..."
2,"Doctor: Hello, how can I help you today? Patie...","{'symptoms': ['fatigue', 'night sweats', 'weig..."


## Batch processing + throughput

Прогоняем подвыборку и считаем приблизительный throughput: generated tokens/sec.

In [15]:
texts = df["dialogue"].tolist()
res0 = extract_entities_for_texts(loaded0, texts, batch_size=BATCH_SIZE)
res0.attrs

IE Qwen/Qwen2.5-1.5B-Instruct (none): 100%|██████████| 30/30 [01:27<00:00,  2.93s/it]


{'generated_tokens': 4725,
 'generation_time_sec': 87.78645369100013,
 'tokens_per_sec': 53.823794006209035}

In [16]:
def current_process_memory_mb() -> float:
    proc = psutil.Process(os.getpid())
    return proc.memory_info().rss / (1024 ** 2)

mem_mb = current_process_memory_mb()
gpu_mem = None
if torch.cuda.is_available():
    gpu_mem = {
        "allocated_mb": torch.cuda.memory_allocated() / (1024 ** 2),
        "reserved_mb": torch.cuda.memory_reserved() / (1024 ** 2),
    }
mem_mb, gpu_mem

(3932.234375, {'allocated_mb': 2953.52734375, 'reserved_mb': 3510.0})

## Сравнение 2–3 моделей

Сравним throughput и стабильность JSON-парсинга. В целях времени можно уменьшить `N_DEMO_ROWS`.

In [17]:
def json_parse_rate(df_out: pd.DataFrame) -> float:
    ok = df_out["parsed"].apply(lambda x: isinstance(x, dict)).sum()
    return ok / max(len(df_out), 1)

compare_rows = []
for mid in MODEL_CANDIDATES[:2]:
    loaded = load_llm(mid, quantization="none")
    out = extract_entities_for_texts(loaded, texts[:60], batch_size=BATCH_SIZE)
    compare_rows.append({
        "model": mid,
        "quant": "none",
        "tokens_per_sec": out.attrs["tokens_per_sec"],
        "parse_rate": json_parse_rate(out),
    })

pd.DataFrame(compare_rows).sort_values(["tokens_per_sec"], ascending=False)

Loading weights: 100%|██████████| 195/195 [00:01<00:00, 123.92it/s, Materializing param=model.norm.weight]                              
Some parameters are on the meta device because they were offloaded to the cpu.
IE microsoft/Phi-3-mini-4k-instruct (none): 100%|██████████| 15/15 [12:59<00:00, 51.95s/it]


,model,quant,tokens_per_sec,parse_rate
0,Qwen/Qwen2.5-1.5B-Instruct,none,55.815628,0.366667
1,microsoft/Phi-3-mini-4k-instruct,none,4.093919,0.816667


## Quantized vs full precision (4-bit)

Этот блок запускается только если есть CUDA и установлен `bitsandbytes`.

In [18]:
quant_results = None
if device == "cuda":
    try:
        loaded_full = load_llm(MODEL_CANDIDATES[0], quantization="none")
        out_full = extract_entities_for_texts(loaded_full, texts[:80], batch_size=BATCH_SIZE)

        loaded_q4 = load_llm(MODEL_CANDIDATES[0], quantization="4bit")
        out_q4 = extract_entities_for_texts(loaded_q4, texts[:80], batch_size=BATCH_SIZE)

        quant_results = pd.DataFrame([
            {"model": MODEL_CANDIDATES[0], "mode": "full", "tokens_per_sec": out_full.attrs["tokens_per_sec"], "parse_rate": json_parse_rate(out_full)},
            {"model": MODEL_CANDIDATES[0], "mode": "4bit", "tokens_per_sec": out_q4.attrs["tokens_per_sec"], "parse_rate": json_parse_rate(out_q4)},
        ])
    except Exception as e:
        print("4-bit block skipped:", repr(e))

quant_results

IE Qwen/Qwen2.5-1.5B-Instruct (4bit): 100%|██████████| 20/20 [13:17:50<00:00, 2393.53s/it]


,model,mode,tokens_per_sec,parse_rate
0,Qwen/Qwen2.5-1.5B-Instruct,full,55.977324,0.3625
1,Qwen/Qwen2.5-1.5B-Instruct,4bit,0.077189,0.2000


## Оценка качества (precision/recall) на small gold

В медицинских диалогах часто нет готовых разметок под нужные классы. Для демонстрации делаем небольшой вручную размеченный synthetic-набор и считаем precision/recall/F1.

Метрика упрощённая: сравниваем нормализованные строки (lower + trim), без лемматизации и синонимов.

In [19]:
GOLD = [
    {
        "text": "Patient: I have a sore throat and fever for three days. Doctor: Start ibuprofen 200 mg twice a day for 5 days.",
        "gold": {
            "symptoms": ["sore throat", "fever"],
            "diagnoses": [],
            "medications": ["ibuprofen"],
            "dosages": ["200 mg", "twice a day"],
            "durations": ["three days", "5 days"],
            "side_effects": [],
        },
    },
    {
        "text": "Patient: Metformin gives me nausea. Doctor: Reduce metformin to 500 mg daily for 2 weeks and monitor.",
        "gold": {
            "symptoms": ["nausea"],
            "diagnoses": [],
            "medications": ["metformin"],
            "dosages": ["500 mg", "daily"],
            "durations": ["2 weeks"],
            "side_effects": ["nausea"],
        },
    },
    {
        "text": "Patient: I have chest tightness and shortness of breath. Doctor: This could be asthma. Use albuterol inhaler as needed.",
        "gold": {
            "symptoms": ["chest tightness", "shortness of breath"],
            "diagnoses": ["asthma"],
            "medications": ["albuterol"],
            "dosages": ["as needed"],
            "durations": [],
            "side_effects": [],
        },
    },
    {
        "text": "Patient: I have headache since yesterday. Doctor: Take paracetamol 1 g every 8 hours for 24 hours.",
        "gold": {
            "symptoms": ["headache"],
            "diagnoses": [],
            "medications": ["paracetamol"],
            "dosages": ["1 g", "every 8 hours"],
            "durations": ["since yesterday", "24 hours"],
            "side_effects": [],
        },
    },
]

def norm_list(xs: List[str]) -> List[str]:
    out = []
    for x in xs:
        x = re.sub(r"\s+", " ", str(x).strip().lower())
        if x:
            out.append(x)
    return out

def prf(tp: int, fp: int, fn: int) -> Dict[str, float]:
    p = tp / (tp + fp) if (tp + fp) else 0.0
    r = tp / (tp + fn) if (tp + fn) else 0.0
    f1 = (2 * p * r / (p + r)) if (p + r) else 0.0
    return {"precision": p, "recall": r, "f1": f1}

def evaluate_on_gold(loaded: LoadedModel) -> pd.DataFrame:
    texts = [g["text"] for g in GOLD]
    out = extract_entities_for_texts(loaded, texts, batch_size=2)

    # aggregate counts
    per_key = {k: {"tp": 0, "fp": 0, "fn": 0} for k in SCHEMA_KEYS}

    for i, g in enumerate(GOLD):
        pred = out.iloc[i]["parsed"] or {k: [] for k in SCHEMA_KEYS}
        for k in SCHEMA_KEYS:
            gold_set = set(norm_list(g["gold"].get(k, [])))
            pred_set = set(norm_list(pred.get(k, [])))
            tp = len(gold_set & pred_set)
            fp = len(pred_set - gold_set)
            fn = len(gold_set - pred_set)
            per_key[k]["tp"] += tp
            per_key[k]["fp"] += fp
            per_key[k]["fn"] += fn

    rows = []
    for k in SCHEMA_KEYS:
        counts = per_key[k]
        scores = prf(counts["tp"], counts["fp"], counts["fn"])
        rows.append({"label": k, **counts, **scores})

    # micro average
    tp = sum(per_key[k]["tp"] for k in SCHEMA_KEYS)
    fp = sum(per_key[k]["fp"] for k in SCHEMA_KEYS)
    fn = sum(per_key[k]["fn"] for k in SCHEMA_KEYS)
    micro = prf(tp, fp, fn)
    rows.append({"label": "micro", "tp": tp, "fp": fp, "fn": fn, **micro})

    return pd.DataFrame(rows)

gold_scores = evaluate_on_gold(loaded0)
gold_scores

IE Qwen/Qwen2.5-1.5B-Instruct (none): 100%|██████████| 2/2 [00:07<00:00,  3.86s/it]


,label,tp,fp,fn,precision,recall,f1
0,symptoms,4,0,2,1.000000,0.666667,0.800000
1,diagnoses,1,0,0,1.000000,1.000000,1.000000
2,medications,0,2,4,0.000000,0.000000,0.000000
3,dosages,0,0,7,0.000000,0.000000,0.000000
4,durations,0,0,5,0.000000,0.000000,0.000000
5,side_effects,0,0,1,0.000000,0.000000,0.000000
6,micro,5,2,19,0.714286,0.208333,0.322581


## Сохранение результатов

Сохраним распарсенные ответы для дальнейшего анализа.

In [20]:
out_dir = "../artifacts"
os.makedirs(out_dir, exist_ok=True)

# Разворачиваем parsed в колонки
def unpack_parsed(row):
    p = row["parsed"] or {k: [] for k in SCHEMA_KEYS}
    return {k: json.dumps(p.get(k, []), ensure_ascii=False) for k in SCHEMA_KEYS}

flat = pd.concat([res0.drop(columns=["parsed"]).reset_index(drop=True), res0.apply(unpack_parsed, axis=1, result_type="expand")], axis=1)
path_csv = os.path.join(out_dir, "ie_results_model0.csv")
flat.to_csv(path_csv, index=False)
path_csv

'../artifacts/ie_results_model0.csv'